In [2]:
!py -m pip install yfinance

  Obtaining dependency information for yfinance from https://files.pythonhosted.org/packages/1e/60/462859de757ac56830824da7e8cf314b8b0321af5853df867c84cd6c2128/yfinance-1.2.0-py2.py3-none-any.whl.metadata
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for frozendict>=2.3.4 from https://files.pythonhosted.org/packages/38/74/f94141b38a51a553efef7f510fc213894161ae49b88bffd037f8d2a7cb2f/frozendict-2.4.7-py3-none-any.whl.metadata
  Obtaining dependency information for peewee>=3.16.2 from https://files.pythonhosted.org/packages/2d/31/93950b2c7145ea10aa454397ffa308c9aadc98dcb4103b676396571bfadd/peewee-4.0.3-py3-none-any.whl.metadata
  Obtaining dependency information for curl_cffi<0.14,>=0.7 from https://files.pythonhosted.org/packages/6d/e4/15a253f9b4bf8d008c31e176c162d2704a7e0c5e24d35942f759df107b68/curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata
   ---------------------------------------- 0.0/130.2 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

## Monthly data processing

### Nasdaq dataset dowload

In [ ]:
import yfinance as yf

# NASDAQ Composite
nasdaq = yf.download("^IXIC", start="1990-01-01", end="2025-12-31")

# Save dataset
nasdaq.to_csv("data/macro/nasdaq_1990_2025.csv")

In [3]:
df = pd.read_excel(r"D:\mycode\Fintech-agent\data\macro\trade balance monthly.xlsx", sheet_name="Query Results")
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Year": "year",
    "Month": "month",
    df.columns[-1]: "trade_balance"
})

df = df.dropna(subset=["year", "month"])

df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-" +
    df["month"].astype(str) + "-01"
) + pd.offsets.MonthEnd(0)

df = df[["date", "trade_balance"]].sort_values("date")

output_path = r"data/macro/Trade balance monthly.csv"
df.to_csv(output_path, index=False)

print(f"Converted successfully → {output_path}")

Converted successfully → data/macro/Trade balance monthly.csv


### merge macro data to monthly order

In [2]:
def to_month_end(series):
    return series.dt.to_period("M").dt.to_timestamp("M")

In [3]:
wti = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\Crude oil price.csv", skiprows=4)
wti.columns = ["date", "wti_price"]
wti["date"]      = pd.to_datetime(wti["date"], format="%b %Y")
wti["wti_price"] = pd.to_numeric(wti["wti_price"], errors="coerce")
wti["date"]      = to_month_end(wti["date"])
wti = wti.sort_values("date").reset_index(drop=True)

In [4]:
fed = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\Federal Funds Effective(Interest) Rate.csv",
                  parse_dates=["observation_date"])
fed = fed.rename(columns={"observation_date": "date", "FEDFUNDS": "fedfunds"})
fed["fedfunds"] = pd.to_numeric(fed["fedfunds"], errors="coerce")
fed["date"]     = to_month_end(fed["date"])

In [5]:
nfci = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\financial conditions index.csv")
nfci = nfci.rename(columns={
    "Friday_of_Week":        "date",
    "NFCI":                  "nfci",
    "ANFCI":                 "anfci",
    "Risk":                  "nfci_risk",
    "Credit":                "nfci_credit",
    "Leverage":              "nfci_leverage",
    "Nonfinancial_Leverage": "nfci_nonfinancial_leverage"
})
nfci["date"] = pd.to_datetime(nfci["date"], format="%m/%d/%Y")
nfci["date"] = to_month_end(nfci["date"])
nfci = (
    nfci.sort_values("date")
    .groupby("date")
    .last()
    .reset_index()
)

In [6]:
gdp = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\GDP.csv",
                  parse_dates=["observation_date"])
gdp = gdp.rename(columns={"observation_date": "date", "GDP": "gdp"})
gdp["gdp"]  = pd.to_numeric(gdp["gdp"], errors="coerce")
gdp["date"] = to_month_end(gdp["date"])
gdp_monthly = (
    gdp.set_index("date")
    .reindex(pd.date_range(gdp["date"].min(), gdp["date"].max(), freq="M"))
    .ffill()
    .reset_index()
    .rename(columns={"index": "date"})
)

gdp_monthly["gdp_growth"] = gdp_monthly["gdp"].pct_change()
gdp_monthly = gdp_monthly.dropna(subset=["gdp_growth"])

In [7]:
gscpi = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\gscpi.csv",
                    usecols=[0, 1])
gscpi.columns    = ["date", "gscpi"]
gscpi["date"]    = pd.to_datetime(gscpi["date"])
gscpi["gscpi"]   = pd.to_numeric(gscpi["gscpi"], errors="coerce")
gscpi["date"]    = to_month_end(gscpi["date"])

In [8]:
gscpi

,date,gscpi
0,1998-01-31,-1.085513
1,1998-02-28,-0.438600
2,1998-03-31,-0.057875
3,1998-04-30,-0.116631
4,1998-05-31,-0.425408
...,...,...
333,2025-10-31,-0.082195
334,2025-11-30,-0.156695
335,2025-12-31,0.543814
336,2026-01-31,0.415778


In [9]:
ppi = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\PPI monthly.csv",
                  parse_dates=["observation_date"])
ppi = ppi.rename(columns={"observation_date": "date", "PPIACO": "ppi"})
ppi["ppi"]  = pd.to_numeric(ppi["ppi"], errors="coerce")
ppi["date"] = to_month_end(ppi["date"])

In [10]:
treasury = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\Treasury and Inflation monthly.csv",
    parse_dates=["caldt"], dayfirst=True
)
treasury = treasury.rename(columns={"caldt": "date"})
treasury["date"] = to_month_end(treasury["date"])

In [11]:
unrate = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\UNRate.csv")
unrate.columns  = ["date", "unrate"]
unrate["date"]  = pd.to_datetime(unrate["date"], format="%m/%d/%Y")
unrate["unrate"] = pd.to_numeric(unrate["unrate"], errors="coerce")
unrate["date"]  = to_month_end(unrate["date"])

In [12]:
usd = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\US Dollar Index Historical Data.csv")
usd = usd[["Date", "Price"]].rename(columns={"Date": "date", "Price": "usd_index"})
usd["date"]      = pd.to_datetime(usd["date"], format="%m-%d-%Y")
usd["usd_index"] = pd.to_numeric(usd["usd_index"], errors="coerce")
usd["date"]      = to_month_end(usd["date"])
usd = usd.sort_values("date").reset_index(drop=True)

In [13]:
tb = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\trade balance monthly.csv",
                 parse_dates=["date"])
tb["trade_balance"] = pd.to_numeric(tb["trade_balance"], errors="coerce")
tb["date"]          = to_month_end(tb["date"])

In [14]:
dfs = [treasury, wti, fed, nfci, gdp_monthly, ppi, unrate, usd, tb]
 
macro_monthly = dfs[0]
for df in dfs[1:]:
    macro_monthly = pd.merge(macro_monthly, df, on="date", how="outer")
 
macro_monthly = macro_monthly.sort_values("date").reset_index(drop=True)

macro_monthly = macro_monthly[
    (macro_monthly["date"] >= "1990-01-01") &
    (macro_monthly["date"] <= "2024-12-31")
].reset_index(drop=True)
 
macro_monthly.to_csv("D:\mycode\Fintech-agent\data\macro\combined_macro_data_monthly.csv",index=False)

### monthly data

In [15]:
nasdaq = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\nasdaq_1990_2025.csv",
                     parse_dates=["Date"])
nasdaq.set_index("Date", inplace=True)
nasdaq_monthly = nasdaq.resample("M").last().reset_index()

macro_monthly.rename(columns={"date": "Date"}, inplace=True)
monthly_merged = pd.merge(nasdaq_monthly, macro_monthly, on="Date", how="inner")
print(monthly_merged.head())

        Date       Close        High         Low        Open     Volume  \
0 1990-01-31  415.799988  415.899994  411.299988  412.200012  148000000   
1 1990-02-28  425.799988  425.799988  422.600006  423.299988  129400000   
2 1990-03-31  435.500000  436.000000  433.200012  434.700012  126630000   
3 1990-04-30  420.100006  420.100006  417.000000  417.500000  105790000   
4 1990-05-31  459.000000  460.000000  457.500000  458.200012  157260000   

     b30ret    b20ret    b10ret     b7ret  ...  nfci_risk  nfci_credit  \
0 -0.042903 -0.039600 -0.024409 -0.015757  ...  -0.062462     0.681974   
1 -0.005948 -0.002690 -0.002074 -0.001334  ...  -0.095063     0.746810   
2 -0.002646 -0.002265 -0.001702  0.000472  ...  -0.110412     0.775024   
3 -0.030332 -0.027292 -0.017396 -0.011241  ...  -0.059055     0.794931   
4  0.048937  0.048854  0.032827  0.029453  ...  -0.191910     0.771955   

   nfci_leverage  nfci_nonfinancial_leverage       gdp  gdp_growth    ppi  \
0       0.425165           

In [16]:
monthly_merged.to_csv("volatility_dataset_monthly.csv",index=False)

### daily data

In [17]:
macro = pd.read_csv("D:\mycode\Fintech-agent\data\macro\combined_macro_data_monthly.csv", parse_dates=["date"])
macro = macro.sort_values("date")

# Rename macro date column to match
macro.rename(columns={"date": "Date"}, inplace=True)

daily_merged = pd.merge_asof(
    nasdaq,
    macro,
    on="Date",
    direction="forward"
)

print(daily_merged.head())

        Date       Close        High         Low        Open     Volume  \
0 1990-01-02  459.299988  459.299988  452.700012  452.899994  110720000   
1 1990-01-03  460.899994  461.600006  460.000000  461.100006  152660000   
2 1990-01-04  459.399994  460.799988  456.899994  460.399994  147950000   
3 1990-01-05  458.200012  459.399994  457.799988  457.899994  137230000   
4 1990-01-08  458.700012  458.700012  456.500000  457.100006  115500000   

     b30ret  b20ret    b10ret     b7ret  ...  nfci_risk  nfci_credit  \
0 -0.042903 -0.0396 -0.024409 -0.015757  ...  -0.062462     0.681974   
1 -0.042903 -0.0396 -0.024409 -0.015757  ...  -0.062462     0.681974   
2 -0.042903 -0.0396 -0.024409 -0.015757  ...  -0.062462     0.681974   
3 -0.042903 -0.0396 -0.024409 -0.015757  ...  -0.062462     0.681974   
4 -0.042903 -0.0396 -0.024409 -0.015757  ...  -0.062462     0.681974   

   nfci_leverage  nfci_nonfinancial_leverage       gdp  gdp_growth    ppi  \
0       0.425165                    0.8

In [18]:
daily_merged.to_csv("volatility_dataset_daily.csv",index=False)

### Combine nasdaq to macro economic data

In [ ]:
import pandas as pd
import numpy as np

def to_month_end(series):
    return series.dt.to_period("M").dt.to_timestamp("M")

def to_quarter_end(series):
    return series.dt.to_period("Q").dt.to_timestamp("Q")

nasdaq = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\nasdaq_1990_2025.csv",
                     parse_dates=["Date"])
nasdaq = nasdaq.sort_values("Date").reset_index(drop=True)

nasdaq["log_return"] = np.log(nasdaq["Close"] / nasdaq["Close"].shift(1))
nasdaq = nasdaq.dropna(subset=["log_return"])

nasdaq["date"] = to_month_end(nasdaq["Date"])

nasdaq_monthly = (
    nasdaq.groupby("date")
    .agg(
        volatility        = ("log_return", lambda x: x.std() * np.sqrt(252)),
        mean_return       = ("log_return", "mean"),
        cumulative_return = ("log_return", lambda x: np.exp(x.sum()) - 1),
        avg_volume        = ("Volume",     "mean"),
        close_price       = ("Close",      "last"),
        trading_days      = ("Close",      "count")
    )
    .reset_index()
)

print("NASDAQ monthly shape:", nasdaq_monthly.shape)
print(f"Date range: {nasdaq_monthly['date'].min()} → {nasdaq_monthly['date'].max()}")
print(nasdaq_monthly.head(3))

macro = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\combined_macro_data.csv",
                    parse_dates=["date"])
macro["date"] = to_quarter_end(macro["date"])
macro = macro.sort_values("date").reset_index(drop=True)

# Create a monthly date index spanning the full macro range
monthly_index = pd.date_range(
    start = macro["date"].min(),
    end   = macro["date"].max(),
    freq  = "M"
)

macro_monthly = (
    macro
    .set_index("date")
    .reindex(monthly_index)
    .ffill()
    .reset_index()
    .rename(columns={"index": "date"})
)

print("\nMacro monthly (forward-filled) shape:", macro_monthly.shape)
print(macro_monthly.head(6))


merged = pd.merge(nasdaq_monthly, macro_monthly, on="date", how="inner")
merged = merged.sort_values("date").reset_index(drop=True)


merged = merged[
    (merged["date"] >= "1990-01-01") &
    (merged["date"] <= "2024-12-31")
].reset_index(drop=True)

print(f"\nMerged shape:  {merged.shape}")
print(f"Date range:    {merged['date'].min()} → {merged['date'].max()}")
print(f"Nulls:\n{merged.isnull().sum()}")
print(f"\nColumns:\n{merged.columns.tolist()}")
print(f"\nSample:\n{merged.head(3)}")

merged.to_csv(r"D:\mycode\Fintech-agent\data\macro\nasdaq_macro_monthly.csv", index=False)
print("\nSaved to nasdaq_macro_monthly.csv")

NASDAQ monthly shape: (432, 7)
Date range: 1990-01-31 00:00:00 → 2025-12-31 00:00:00
        date  volatility  mean_return  cumulative_return    avg_volume  \
0 1990-01-31    0.140752    -0.004738          -0.094709  1.389100e+08   
1 1990-02-28    0.093086     0.001251           0.024050  1.274168e+08   
2 1990-03-31    0.084232     0.001024           0.022781  1.311568e+08   

   close_price  trading_days  
0   415.799988            21  
1   425.799988            19  
2   435.500000            22  

Macro monthly (forward-filled) shape: (418, 24)
        date    b30ret    b20ret    b10ret     b7ret     b5ret     b2ret  \
0 1990-03-31 -0.051113 -0.044353 -0.028089 -0.016606 -0.009294  0.006020   
1 1990-04-30 -0.051113 -0.044353 -0.028089 -0.016606 -0.009294  0.006020   
2 1990-05-31 -0.051113 -0.044353 -0.028089 -0.016606 -0.009294  0.006020   
3 1990-06-30  0.043260  0.043723  0.032971  0.033665  0.033669  0.028198   
4 1990-07-31  0.043260  0.043723  0.032971  0.033665  0.033669  0

C:\Users\manoj\AppData\Local\Temp\ipykernel_22116\1179995362.py:48: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_index = pd.date_range(


## Quatarly dataset processing

### Convert xls to csv data

In [ ]:
df = pd.read_excel("D:\mycode\Fintech-agent\data\macro\Trade balance.xlsx", sheet_name="Query Results")
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Year": "year",
    "Month": "month",
    df.columns[-1]: "trade_balance"
})

df = df.dropna(subset=["year", "month"])

df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df_q = df[df["month"].isin([3, 6, 9, 12])].copy()

df_q["date"] = pd.to_datetime(
    df_q["year"].astype(str) + "-" +
    df_q["month"].astype(str) + "-01"
) + pd.offsets.MonthEnd(0)


df_q = df_q[["date", "trade_balance"]].sort_values("date")
output_path = r"data/macro/Trade balance.csv"
df_q.to_csv(output_path, index=False)

print(f"Converted successfully → {output_path}")

Converted successfully → data/macro/Trade balance.csv


In [ ]:
df = pd.read_excel("data/macro/gscpi_data.xls", sheet_name = "GSCPI Monthly Data")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date"])
df.to_csv("data/macro/gscpi.csv", index=False)
print(f"Converted!")


### Merge CSVs

In [3]:
def to_quarter_end(series: pd.Series) -> pd.Series:
    return  series.dt.to_period("Q").dt.to_timestamp("Q") 

In [12]:
inflation = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\Treasury and Inflation.csv",
    parse_dates=["caldt"],
    dayfirst=True       
)


inflation["date"] = to_quarter_end(inflation["caldt"])
inflation = inflation.drop(columns=["caldt"])

inflation = (
    inflation
    .groupby("date")
    .last()
    .reset_index()
)

wti = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\Crude oil price.csv",
    skiprows=4,
    parse_dates=["Month"]
)
wti = wti.rename(columns={
    "Cushing OK WTI Spot Price FOB Dollars per Barrel": "wti_price"
})


wti["wti_price"] = pd.to_numeric(wti["wti_price"], errors="coerce")
wti["date"] = to_quarter_end(wti["Month"])
wti = wti.sort_values("date")

wti = (
    wti.groupby("date")
    .agg(wti_price=("wti_price", "mean"))   
    .reset_index()
)

merged = pd.merge(inflation, wti, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)

In [13]:
fed = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\Federal Funds Effective(Interest) Rate.csv",
    parse_dates=["observation_date"]
)
fed = fed.rename(columns={"observation_date": "date", "FEDFUNDS": "fedfunds"})
fed["fedfunds"] = pd.to_numeric(fed["fedfunds"], errors="coerce")
 
fed["date"] = to_quarter_end(fed["date"])
 
fed = (
    fed.groupby("date")
    .agg(fedfunds=("fedfunds", "mean"))
    .reset_index()
)
merged = pd.merge(merged, fed, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)

In [14]:

nfci = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\financial conditions index.csv",
    parse_dates=False
)
 
nfci = nfci.rename(columns={
    "Friday_of_Week":        "date",
    "NFCI":                  "nfci",
    "ANFCI":                 "anfci",
    "Risk":                  "nfci_risk",
    "Credit":                "nfci_credit",
    "Leverage":              "nfci_leverage",
    "Nonfinancial_Leverage": "nfci_nonfinancial_leverage"
})
 
nfci["date"] = pd.to_datetime(nfci["date"], format="%m/%d/%Y")
nfci["date"] = to_quarter_end(nfci["date"])
 
nfci = (
    nfci.groupby("date")
    .agg(
        nfci=("nfci", "mean"),
        anfci=("anfci", "mean"),
        nfci_risk=("nfci_risk", "mean"),
        nfci_credit=("nfci_credit", "mean"),
        nfci_leverage=("nfci_leverage", "mean"),
        nfci_nonfinancial_leverage=("nfci_nonfinancial_leverage", "mean")
    )
    .reset_index()
)
 
merged = pd.merge(merged, nfci, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)
 

In [15]:
gdp = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\GDP.csv", parse_dates=["observation_date"])
gdp = gdp.rename(columns={"observation_date": "date", "GDP": "gdp"})
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")
gdp["date"] = to_quarter_end(gdp["date"])
gdp = gdp.groupby("date").agg(gdp=("gdp", "last")).reset_index()
gdp["gdp_growth"] = gdp["gdp"].pct_change()
merged = pd.merge(merged, gdp, on="date", how="left")
 
ppi = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\PPI.csv", parse_dates=["observation_date"])
ppi = ppi.rename(columns={"observation_date": "date", "PPIACO": "ppi"})
ppi["ppi"] = pd.to_numeric(ppi["ppi"], errors="coerce")
ppi["date"] = to_quarter_end(ppi["date"])
ppi = ppi.groupby("date").agg(ppi=("ppi", "mean")).reset_index()
merged = pd.merge(merged, ppi, on="date", how="left")
 
tb = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\Trade balance.csv", parse_dates=["date"])
tb["trade_balance"] = pd.to_numeric(tb["trade_balance"], errors="coerce")
tb["date"] = to_quarter_end(tb["date"])
tb = tb.groupby("date").agg(trade_balance=("trade_balance", "sum")).reset_index()
merged = pd.merge(merged, tb, on="date", how="left")
 
unrate = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\UNRate.csv")
unrate.columns = ["date", "unrate"]
unrate["date"] = pd.to_datetime(unrate["date"], format="%m/%d/%Y")
unrate["unrate"] = pd.to_numeric(unrate["unrate"], errors="coerce")
unrate["date"] = to_quarter_end(unrate["date"])
unrate = unrate.groupby("date").agg(unrate=("unrate", "mean")).reset_index()
merged = pd.merge(merged, unrate, on="date", how="left")
 
merged = merged.sort_values("date").reset_index(drop=True)

In [16]:
usd = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\US Dollar Index Historical Data.csv")
usd = usd[["Date", "Price"]].rename(columns={"Date": "date", "Price": "usd_index"})
usd["date"] = pd.to_datetime(usd["date"], format="%m-%d-%Y")
usd["usd_index"] = pd.to_numeric(usd["usd_index"], errors="coerce")
usd["date"] = to_quarter_end(usd["date"])
usd = usd.sort_values("date")
usd = usd.groupby("date").agg(usd_index=("usd_index", "mean")).reset_index()
merged = pd.merge(merged, usd, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)
 

In [17]:
merged = merged[
    (merged["date"] >= "1990-01-01") &
    (merged["date"] <= "2024-12-31")
].reset_index(drop=True)
 

In [10]:
housing = pd.read_csv("D:\mycode\Fintech-agent\data\macro\housing.csv", encoding="utf-16",sep="\t")     

housing["date"] = pd.to_datetime(housing["Month of Period End"], format="%B %Y")
housing["date"] = housing["date"].dt.to_period("Q").dt.to_timestamp("Q")
housing = housing.drop(columns=["Region", "Month of Period End"])
 
mom_yoy_cols = [c for c in housing.columns if "MoM" in c or "YoY" in c]
housing = housing.drop(columns=mom_yoy_cols)
def clean_column(series):
    """Remove $, K, % symbols and convert to float"""
    if series.dtype == object:
        series = (series
                  .str.replace("$", "", regex=False)
                  .str.replace("K", "", regex=False)
                  .str.replace("%", "", regex=False)
                  .str.replace(",", "", regex=False)
                  .str.strip())
        return pd.to_numeric(series, errors="coerce")
    return series
 
housing["Median Sale Price"] = (
    housing["Median Sale Price"]
    .str.replace("$", "", regex=False)
    .str.replace("K", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce") * 1000
)

for col in housing.columns:
    if col != "date" and col != "Median Sale Price":
        housing[col] = clean_column(housing[col])

housing_quarterly = housing.groupby("date").mean().reset_index()
merged["date"] = pd.to_datetime(merged["date"])
merged["date"] = merged["date"].dt.to_period("Q").dt.to_timestamp("Q")
merged = pd.merge(merged, housing_quarterly, on="date", how="inner")
 

In [18]:
merged.to_csv("D:\mycode\Fintech-agent\data\macro\combined_macro_data.csv", index=False)
print(f"Saved: data/macro/combined_macro_data.csv")
print(f"Shape: {merged.shape[0]} rows × {merged.shape[1]} columns")
print(f"Date range: {merged['date'].min().date()} → {merged['date'].max().date()}")
print(f"\nColumns:\n{list(merged.columns)}")

Saved: data/macro/combined_macro_data.csv
Shape: 140 rows × 25 columns
Date range: 1990-03-31 → 2024-12-31

Columns:
['date', 'b30ret', 'b20ret', 'b10ret', 'b7ret', 'b5ret', 'b2ret', 'b1ret', 't90ret', 't30ret', 'cpiret', 'wti_price', 'fedfunds', 'nfci', 'anfci', 'nfci_risk', 'nfci_credit', 'nfci_leverage', 'nfci_nonfinancial_leverage', 'gdp', 'gdp_growth', 'ppi', 'trade_balance', 'unrate', 'usd_index']


In [19]:
print(merged.isnull().sum())

date                          0
b30ret                        0
b20ret                        0
b10ret                        0
b7ret                         0
b5ret                         0
b2ret                         0
b1ret                         0
t90ret                        0
t30ret                        0
cpiret                        0
wti_price                     0
fedfunds                      0
nfci                          0
anfci                         0
nfci_risk                     0
nfci_credit                   0
nfci_leverage                 0
nfci_nonfinancial_leverage    0
gdp                           0
gdp_growth                    0
ppi                           0
trade_balance                 0
unrate                        0
usd_index                     0
dtype: int64
